# ML Anomaly Scoring: Isolation Forest & LOF

Everything in `01`-`09` is rule-based: a hypothesis formalized as a boolean condition. This notebook takes the complementary, unsupervised approach - an `IsolationForest` model looks for structure in the data without being told what an anomaly looks like, to check whether the four rule-based clusters found so far are the whole picture or just the most obvious part of it.

Standalone notebook — reloads and prepares the data independently of `01_eda.ipynb`.

In [1]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

df = pd.read_csv('/home/veronika/Anomaly_Hunter_Solidgate/hackathon_int20h_dataset_test.csv')
df['created_at'] = pd.to_datetime(df['created_at'])
df['processed_at'] = pd.to_datetime(df['processed_at'])

## Feature engineering

Both `IsolationForest` and `LocalOutlierFactor` operate on numeric feature vectors, so every earlier lesson about scale and currency carries over: raw IDs (`order_id`, `user_id`, `bank_id`) are excluded (no meaningful magnitude), and `amount` is z-scored within currency rather than used raw - the same correction from `07_anomaly_amount_outliers.ipynb`.

Starting feature set: `latency_sec`, `amount_zscore`, `is_fail`, `is_secured`.

In [2]:
df['latency_sec'] = (df['processed_at'] - df['created_at']).dt.total_seconds()
df['is_fail'] = (df['status'] == 'fail').astype(int)
df['amount_zscore'] = df.groupby('currency')['amount'].transform(lambda x: (x - x.mean()) / x.std())

## Validation strategy: no labels, but four known clusters

There is no `is_anomaly` ground truth in this dataset, so precision/recall can't be computed directly. Instead, the four already-confirmed clusters (`bank_id == 777`, the psp_gamma latency window, the psp_beta over-refund rows, the `09` fail-refund rows) serve as a known-positive sanity check: after fitting, their `decision_function` scores are compared against the dataset baseline. If a cluster's scores sit well below the global distribution, the model is picking up on the same structure found manually. If not, the feature set is missing whatever made that cluster stand out.

In [3]:
known_bank777 = df['bank_id'] == 777
known_gamma = (
    (df['psp_id'] == 'psp_gamma') &
    (df['created_at'] >= '2025-05-12') & (df['created_at'] <= '2025-05-17 23:59:59')
)
known_beta = df['refunded_amount'] > df['amount']
known_failrefund = df['has_refund'] & (df['status'] == 'fail')
known = known_bank777 | known_gamma | known_beta | known_failrefund

clusters = {
    'bank_id==777': known_bank777,
    'psp_gamma window': known_gamma,
    'psp_beta over-refund': known_beta,
    'fail_refund': known_failrefund,
}

def sanity_check(scores):
    print('global:', scores.describe()[['mean', '50%', 'min', 'max']].to_dict())
    for name, mask in clusters.items():
        print(name, scores[mask].describe()[['mean', '50%', 'min', 'max']].to_dict())

## First pass: four features, no refund information

In [4]:
X1 = df[['latency_sec', 'amount_zscore', 'is_fail', 'is_secured']]
model_v1 = IsolationForest(random_state=42, contamination='auto').fit(X1)
scores_v1 = pd.Series(model_v1.decision_function(X1), index=df.index)
sanity_check(scores_v1)

global: {'mean': 0.021733970647927068, '50%': 0.03679350581126534, 'min': -0.28754484078789055, 'max': 0.08876018309115774}
bank_id==777 {'mean': 0.03714648821784189, '50%': 0.05372336929137117, 'min': -0.2162842026830979, 'max': 0.08169777854601301}
psp_gamma window {'mean': -0.1352205183651056, '50%': -0.13124691291874824, 'min': -0.28754484078789055, 'max': -0.05627502330952272}
psp_beta over-refund {'mean': 0.02524676095597313, '50%': 0.03916430155920303, 'min': -0.226742766435165, 'max': 0.08876018309115774}
fail_refund {'mean': -0.03898647377364973, '50%': -0.010235300403996672, 'min': -0.2348216876639072, 'max': 0.08594047110816383}


**Finding:** `IsolationForest`'s `decision_function` runs on the convention *lower = more anomalous*. Against a global mean of 0.022:
- **psp_gamma is clearly detected** (mean -0.135, entirely below the global range) - a 1-2 hour delay is an extreme value on `latency_sec` for any individual row, a textbook **point anomaly**.
- **`bank_id == 777` is not detected at all** (mean 0.037, slightly *above* global) - its defining trait was that all 635 rows share an identical 5-second latency, a value that is completely unremarkable on its own (normal fast transactions run 2-10 seconds). The anomaly is the zero variance *within the group*, a **collective anomaly** - invisible to a model that scores each row independently.
- **psp_beta's over-refund rows are not detected** (mean 0.025, essentially global baseline) and **fail_refund is only partially separated** (mean -0.039) - neither cluster's defining signal (the relationship between `amount` and `refunded_amount`) is represented anywhere in this feature set.

## Second pass: adding a refund-aware feature

Add `refund_gap = refunded_amount - amount`, the exact quantity that defines both remaining clusters.

In [5]:
df['refund_gap'] = df['refunded_amount'] - df['amount']

X2 = df[['latency_sec', 'amount_zscore', 'is_fail', 'is_secured', 'refund_gap']]
model_v2 = IsolationForest(random_state=42, contamination='auto').fit(X2)
scores_v2 = pd.Series(model_v2.decision_function(X2), index=df.index)
sanity_check(scores_v2)

global: {'mean': 0.05886817389565755, '50%': 0.08282509403298216, 'min': -0.3113423576377554, 'max': 0.12621691717383038}
bank_id==777 {'mean': 0.06690341590202883, '50%': 0.09709498158551205, 'min': -0.22169363737031544, 'max': 0.1180119157458388}
psp_gamma window {'mean': -0.10315304438295002, '50%': -0.09090354867679873, 'min': -0.3113423576377554, 'max': -0.04326926806271225}
psp_beta over-refund {'mean': 0.06362755733222003, '50%': 0.08066594115495623, 'min': -0.15821878005918122, 'max': 0.11109776076961975}
fail_refund {'mean': -0.0005586074202737831, '50%': 0.023780793810327105, 'min': -0.25929065499454507, 'max': 0.11647127970572135}


**Finding:** psp_beta's separation barely moves (mean 0.064 vs. a global mean of 0.059) - `refund_gap` was added **raw, in each transaction's local currency**, unlike `amount_zscore`. The same currency-mixing trap from `01_eda.ipynb` §10 and `07_anomaly_amount_outliers.ipynb` applies here too: a currency with a larger numeric scale (e.g. UAH) produces a larger raw `refund_gap` for an equally-sized relative refund than a smaller-scale currency (e.g. GBP), so the feature doesn't mean the same thing across currencies.

## Third pass: normalizing the refund feature within currency

In [6]:
df['refund_gap_zscore'] = df.groupby('currency')['refund_gap'].transform(lambda x: (x - x.mean()) / x.std())

X3 = df[['latency_sec', 'amount_zscore', 'is_fail', 'is_secured', 'refund_gap_zscore']]
model = IsolationForest(random_state=42, contamination='auto').fit(X3)
df['if_scores'] = model.decision_function(X3)
sanity_check(df['if_scores'])

global: {'mean': 0.028259828151077904, '50%': 0.04638501301125414, 'min': -0.2998960606250983, 'max': 0.09972893465074217}
bank_id==777 {'mean': 0.0345706381298238, '50%': 0.05176323999757437, 'min': -0.2406619511579393, 'max': 0.08549958718243406}
psp_gamma window {'mean': -0.11500672152438475, '50%': -0.10378150539285136, 'min': -0.2998960606250983, 'max': -0.053369704409430896}
psp_beta over-refund {'mean': -0.026816811323900142, '50%': -0.013351928532950974, 'min': -0.18012348981282755, 'max': 0.03764412230096331}
fail_refund {'mean': -0.02613576948619423, '50%': -0.013955501722871966, 'min': -0.1941679779706631, 'max': 0.0697648883316217}


**Finding:** with the currency-normalized feature, both remaining clusters now separate cleanly - psp_beta (mean -0.027, median -0.013) and fail_refund (mean -0.026, median -0.014) both sit below the global mean (0.028). `bank_id == 777` remains undetected (mean 0.035, essentially at baseline), consistent with it being a collective rather than a point anomaly - no feature engineering fixes that; it would need a model that reasons about groups, not individual rows.

## Local Outlier Factor - a second method, and where it breaks

`LocalOutlierFactor` scores each point by comparing its local density to its neighbors' - unlike `IsolationForest`'s random-split approach, it depends on genuine, non-degenerate distances between points.

In [7]:
lof = LocalOutlierFactor(n_neighbors=20)
lof_pred = lof.fit_predict(X3)
lof_scores = pd.Series(lof.negative_outlier_factor_, index=df.index)

print(pd.Series(lof_pred).value_counts())
print(lof_scores.describe())

 1    992292
-1      7708
Name: count, dtype: int64
count    1.000000e+06
mean    -8.045371e+05
std      3.653931e+07
min     -7.011793e+09
25%     -1.000000e+00
50%     -1.000000e+00
75%     -1.000000e+00
max     -9.161438e-01
dtype: float64


/home/veronika/.local/lib/python3.10/site-packages/sklearn/neighbors/_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


**Finding:** the score distribution is not usable - mean around -800,000, a minimum near **-7 billion**, while the 25th/50th/75th percentiles all sit at exactly -1.0. sklearn raises `UserWarning: Duplicate values are leading to incorrect results`, and the cause is structural: several features here are near-discrete (`latency_sec` only takes ~9 values for normal transactions, `is_fail`/`is_secured` are binary, `refund_gap_zscore` is one repeated constant for the majority of rows with no refund at all) - so thousands of rows share an identical feature vector. LOF's density estimate divides by distance-to-neighbor, and when that distance is zero, the ratio blows up. `IsolationForest` is unaffected by this because random-split trees don't depend on exact distances - the same duplicate-value sensitivity that broke IQR-based quantiles for the `recurring` price ladder in `07_anomaly_amount_outliers.ipynb` resurfaces here, breaking a different (distance-based) method instead. **LOF's output is not used for anything below** - it isn't reliable with this feature set.

## Looking beyond the four known clusters

Using the validated `IsolationForest` scores (with `refund_gap_zscore`), take the most anomalous 1% of the dataset (roughly matching the combined size of the four known clusters) and check how much of it is new territory.

In [8]:
threshold = df['if_scores'].quantile(0.01)
flagged = df['if_scores'] <= threshold
new = flagged & ~known

print(f"Flagged: {flagged.sum():,} | Known overlap: {(flagged & known).sum():,} | New: {new.sum():,}")

new_df = df[new]
print(new_df['psp_id'].value_counts())
print(new_df['currency'].value_counts())
print(new_df['status'].value_counts())
print(new_df['bank_id'].value_counts().head(8))

Flagged: 10,006 | Known overlap: 424 | New: 9,582
psp_id
psp_gamma    3372
psp_beta     3354
psp_alpha    2856
Name: count, dtype: int64
currency
USD    4680
CAD    1426
EUR    1153
PLN     640
GBP     621
UAH     563
MXN     499
Name: count, dtype: int64
status
fail       5686
success    3896
Name: count, dtype: int64
bank_id
49    236
2     225
30    224
3     224
42    218
16    215
45    212
31    210
Name: count, dtype: int64


**Finding:** of 10,006 flagged rows, only 424 overlap the four known clusters - the rest (9,582 rows) sit outside them. Profiling this residual:
- `psp_id` is close to evenly split (psp_gamma 3,372, psp_beta 3,354, psp_alpha 2,856) - no single provider dominates.
- `bank_id` is flat across the board (210-236 per bank in the top 8 of ~49) - no bank stands out.
- `currency` roughly tracks the dataset's baseline shares (USD 46.8%, CAD 14.3%, EUR 11.5%, the rest 5-6% each) - no currency-scale artifact left, confirming the `refund_gap_zscore` fix worked.
- `status` skews toward `fail` (56.8% vs. a ~47.5% dataset baseline) - a real but moderate tilt, not a sharp signature like the four confirmed clusters.

**Conclusion:** no fifth coherent cluster emerges - the "new" 9,582 rows look like a diffuse mix of borderline cases (elevated on one feature or another) rather than a single pattern with its own fingerprint, the way each of the four rule-based clusters had one. Given this, the residual is treated as unconfirmed rather than folded into `is_anomaly`.

## Conclusions

| Cluster | Detected by `IsolationForest`? | Why |
|---|---|---|
| `bank_id == 777` | No | Collective anomaly (zero variance within the group) - not visible to a row-independent model, regardless of feature choice |
| psp_gamma latency | Yes, cleanly | Point anomaly - directly captured by `latency_sec` |
| psp_beta over-refund | Yes, after currency-normalizing `refund_gap` | Required a refund-aware, currency-corrected feature |
| fail_refund (`09`) | Yes, after the same fix | Same as above |

`LocalOutlierFactor` was not usable on this feature set - near-duplicate feature vectors (from several near-discrete columns) break its distance-based density estimate, confirmed by sklearn's own duplicate-value warning and an unusable score range.

Scanning the top 1% most anomalous rows by `IsolationForest` score surfaces 9,582 rows outside the four known clusters, but they show no shared fingerprint (balanced across `psp_id`, `bank_id`, and `currency`) - read as diffuse noise rather than a fifth cluster. **Recommendation for the final `is_anomaly` column:** rely on the four rule-based flags as the primary source of truth; treat the model as a validation tool that confirms three of the four clusters and correctly cannot see the fourth, rather than as a source of additional confirmed labels.